# `get_search_trend` 사용법

**검색어트렌드** — 네이버 검색창에 그 단어를 친 횟수의 추이.

`get_trend`(쇼핑인사이트)는 *네이버쇼핑에서 상품을 클릭한 수*다. 둘은 다른 행동이고,
**둘을 나란히 놓는 것**이 이 함수를 붙인 이유다.

아래 출력은 실제로 돌려 받은 값이다 (2026-10-22). 날짜가 지나면 숫자는 달라진다.

In [2]:
import sys
sys.path.insert(0, "src")

import pandas as pd
from fashion_trend.naver import get_search_trend, get_trend, to_wide

## 1. 기본

인자는 `get_trend` 와 거의 같다. 반환 형태도 같아서 `to_wide` 가 그대로 먹는다.

In [3]:
df = get_search_trend(["가디건", "니트", "코트"],
                      weeks=12, gender="f", ages=["10", "20"])

to_wide(df).tail(3).round(2)

keyword,가디건,니트,코트
period,,,
2026-09-28,84.16,80.63,63.61
2026-10-05,82.59,95.16,83.77
2026-10-12,76.83,100.00,72.12


```
keyword       가디건      니트     코트
2026-09-28  84.16   80.63  63.61
2026-10-05  82.59   95.16  83.77
2026-10-12  76.83  100.00  72.12
```

`ratio` 는 여기서도 **요청 안에서 최댓값을 100으로 둔 상대지수**다.
배치가 다르면 직접 비교하면 안 되는 것도 똑같다.

## 2. `get_trend` 와 뭐가 다른가

| | `get_trend` (쇼핑인사이트) | `get_search_trend` (검색어트렌드) |
|---|---|---|
| 재는 것 | 네이버쇼핑 상품 **클릭** | 검색창 **검색** |
| 카테고리 | 패션의류로 한정 가능 | **없음 — 네이버 전체** |
| 묶기 | 키워드 1개씩만 | 한 그룹에 **최대 20개 합산** |
| 요청당 | 키워드 5개 | 그룹 5개 |
| 연령 | `"10"`, `"20"` | `"1"~"11"` 구간 코드 |

카테고리가 없다는 게 약점이다. "코트" 가 외투인지 법정인지 구분되지 않는다.
대신 묶기가 되는 게 강점이다 — 바로 아래.

## 3. 표기 변형 묶기  ★

쇼핑인사이트에서는 "가디건" 과 "카디건" 이 완전히 다른 키워드로 잡히고 합칠 수가 없었다.
여기서는 한 그룹에 넣으면 **합산해서** 돌려준다. 딕셔너리로 넘기면 된다.

In [4]:
변형 = get_search_trend({
    "가디건(단독)":       ["가디건"],
    "가디건+카디건":       ["가디건", "카디건"],
    "스웨이드자켓":        ["스웨이드자켓"],
    "스웨이드자켓+재킷":    ["스웨이드자켓", "스웨이드재킷"],
}, weeks=12)

w = to_wide(변형)
print("합산 효과:", round((w["가디건+카디건"] / w["가디건(단독)"]).mean(), 3), "배")
w.tail(3).round(2)

합산 효과: 1.073 배


keyword,가디건(단독),가디건+카디건,스웨이드자켓,스웨이드자켓+재킷
period,,,,
2026-09-28,54.61,57.00,78.49,78.91
2026-10-05,52.52,54.95,99.50,100.00
2026-10-12,44.81,47.13,85.80,86.10


```
합산 효과: 1.073 배

keyword     가디건(단독)  가디건+카디건  스웨이드자켓  스웨이드자켓+재킷
2026-09-28    54.61    57.00   78.49      78.91
2026-10-05    52.52    54.95   99.50     100.00
2026-10-12    44.81    47.13   85.80      86.10
```

"가디건" 만 보면 **7% 를 놓친다.** 표기 변형이 많은 키워드일수록 차이가 커진다.
한 그룹에 20개까지 들어간다.

## 4. 연령 코드가 다르다  ★ 주의

네이버가 여기서는 10년 단위가 아니라 구간 코드를 쓴다.

| 코드 | 나이 | | 코드 | 나이 |
|---|---|---|---|---|
| 1 | 0~12 | | 7 | 40~44 |
| 2 | 13~18 | | 8 | 45~49 |
| 3 | 19~24 | | 9 | 50~54 |
| 4 | 25~29 | | 10 | 55~59 |
| 5 | 30~34 | | 11 | 60~ |
| 6 | 35~39 | | | |

쇼핑인사이트와 같은 인자로 쓸 수 있도록 `ages=["10","20"]` 을 주면 **안에서 변환**한다.

In [ ]:
from fashion_trend.naver import _to_buckets, AGE_BUCKETS

print(_to_buckets(["10", "20"]))        # 10대·20대 → 구간 코드
print([AGE_BUCKETS[c] for c in _to_buckets(["10", "20"])])

```
['2', '3', '4']
['13~18', '19~24', '25~29']
```

**구간이 10년 단위로 안 떨어진다.** 코드 `3` 이 19~24세라 **19세가 20대 쪽으로 샌다.**
정확히 맞추고 싶으면 코드를 직접 넘긴다.

```python
get_search_trend(["가디건"], ages_raw=["2", "3"])   # 13~24세
```

## 5. 그룹이 5개를 넘으면

자동으로 나눠 호출한다. 나뉜 뒤에는 숫자를 비교하면 안 되는 것도 `get_trend` 와 같다.

In [ ]:
big = get_search_trend(["가디건", "니트", "코트", "패딩", "셔츠", "청바지", "원피스"],
                       weeks=8)
big.groupby("batch")["keyword"].unique()

```
batch
1    [가디건, 니트, 셔츠, 코트, 패딩]
2               [원피스, 청바지]
```

## 6. 괴리 신호 — 이걸 보려고 붙였다  ★

검색과 쇼핑 클릭이 **같이 움직이는지**가 핵심이다.

| 검색 | 쇼핑 클릭 | 해석 |
|---|---|---|
| ↑ | ↑ | 진짜 수요 |
| ↑ | → | 화제성뿐, 구매로 안 이어짐 → **사입 조심** |
| → | ↑ | 이미 살 마음 먹은 사람들 → 늦었을 수 있음 |

In [ ]:
def 괴리(keywords, weeks=12, **seg):
    s = to_wide(get_search_trend(keywords, weeks=weeks, **seg))
    c = to_wide(get_trend(keywords, weeks=weeks, **seg))
    rows = []
    for k in keywords:
        g = lambda t: t[k].iloc[-4:].mean() / t[k].iloc[-8:-4].mean() - 1
        rows.append({"keyword": k, "검색": g(s), "쇼핑클릭": g(c)})
    out = pd.DataFrame(rows).set_index("keyword")
    out["차이"] = out["쇼핑클릭"] - out["검색"]
    return out

괴리(["가디건", "패딩"], gender="f", ages=["10", "20"]).round(3)

```
           검색   쇼핑클릭     차이
가디건    0.102   0.023 -0.079
패딩     0.592   1.424  0.832
```

가디건은 검색이 늘어도 클릭이 안 따라온다. 패딩은 클릭이 검색보다 훨씬 빠르게 는다.

단, 두 수치는 **서로 다른 요청**에서 왔으므로 크기를 직접 비교하면 안 된다.
비교할 수 있는 건 각자의 **증감률**뿐이다 — 위 함수가 그래서 비율만 쓴다.

---
## 인자 정리

```python
get_search_trend(
    groups,                # ["가디건","니트"]  또는  {"가디건": ["가디건","카디건"]}
    weeks=52,
    start=None, end=None,
    time_unit="week",      # "date" | "week" | "month"
    gender=None,           # "m" | "f"
    ages=None,             # ["10","20"] — 내부에서 구간 코드로 변환
    ages_raw=None,         # ["2","3"] — 네이버 코드 직접. 주면 ages 를 무시
    device=None,           # "pc" | "mo"
    env_path=".env",
)
```

### 기억할 것

1. 카테고리가 없다. **네이버 전체 검색**이라 동음이의어가 섞인다
2. 그룹 5개 · 그룹당 키워드 20개
3. 연령 코드가 쇼핑인사이트와 **호환되지 않는다**. 19세는 20대로 샌다
4. 상대지수다. 배치가 다르면 증감률만 비교한다